# Phase 2: Data Cleaning and Preparation

This notebook removes duplicates, handles missing values, standardizes dates, checks invalid sales values, creates the required columns, and saves the cleaned dataset.

## 1. Import libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## 2. Set the folder paths

In [2]:
current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

raw_data_folder = project_folder / 'data' / 'raw'
processed_data_folder = project_folder / 'data' / 'processed'
processed_data_folder.mkdir(parents=True, exist_ok=True)

## 3. Load the datasets

In [3]:
customers = pd.read_csv(raw_data_folder / 'Customers.csv', encoding='cp1252', keep_default_na=False, na_values=[''])
products = pd.read_csv(raw_data_folder / 'Products.csv', keep_default_na=False, na_values=[''])
sales = pd.read_csv(raw_data_folder / 'Sales.csv', keep_default_na=False, na_values=[''])
stores = pd.read_csv(raw_data_folder / 'Stores.csv', keep_default_na=False, na_values=[''])
exchange_rates = pd.read_csv(raw_data_folder / 'Exchange_Rates.csv', keep_default_na=False, na_values=[''])

print('Datasets loaded successfully.')

Datasets loaded successfully.


## 4. Remove duplicate records

In [4]:
customers = customers.drop_duplicates().copy()
products = products.drop_duplicates().copy()
sales = sales.drop_duplicates().copy()
stores = stores.drop_duplicates().copy()
exchange_rates = exchange_rates.drop_duplicates().copy()

print('Duplicate records removed.')

Duplicate records removed.


## 5. Handle missing values

Blank delivery dates are kept empty because physical store purchases do not require delivery. The online store has no physical area, so its missing square meters value is changed to zero.

In [5]:
online_store = stores['StoreKey'] == 0
stores.loc[online_store, 'Square Meters'] = stores.loc[online_store, 'Square Meters'].fillna(0)

print('Remaining missing values in Stores:')
display(stores.isna().sum())

print('Missing delivery dates kept as not applicable:', sales['Delivery Date'].isna().sum())

Remaining missing values in Stores:


StoreKey         0
Country          0
State            0
Square Meters    0
Open Date        0
dtype: int64

Missing delivery dates kept as not applicable: 49719


## 6. Standardize date columns

In [6]:
sales['Order Date'] = pd.to_datetime(sales['Order Date'], errors='coerce')
sales['Delivery Date'] = pd.to_datetime(sales['Delivery Date'], errors='coerce')
customers['Birthday'] = pd.to_datetime(customers['Birthday'], errors='coerce')
stores['Open Date'] = pd.to_datetime(stores['Open Date'], errors='coerce')
exchange_rates['Date'] = pd.to_datetime(exchange_rates['Date'], errors='coerce')

print('Date columns converted to datetime format.')

Date columns converted to datetime format.


## 7. Clean product cost and price columns

In [7]:
products['Unit Cost USD'] = (
    products['Unit Cost USD']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

products['Unit Price USD'] = (
    products['Unit Price USD']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

display(products[['Unit Cost USD', 'Unit Price USD']].head())

,Unit Cost USD,Unit Price USD
0,6.62,12.99
1,6.62,12.99
2,7.40,14.52
3,11.00,21.57
4,11.00,21.57


## 8. Check and remove invalid sales values

In [8]:
invalid_quantity = sales['Quantity'] <= 0
invalid_cost = products['Unit Cost USD'] < 0
invalid_price = products['Unit Price USD'] <= 0

print('Invalid sales quantities:', invalid_quantity.sum())
print('Invalid product costs:', invalid_cost.sum())
print('Invalid product prices:', invalid_price.sum())

sales = sales[~invalid_quantity].copy()
products = products[~invalid_cost & ~invalid_price].copy()

Invalid sales quantities: 0
Invalid product costs: 0
Invalid product prices: 0


## 9. Combine the datasets

The tables are joined using their common keys so the cleaned file contains sales, product, customer, store, and exchange-rate information.

In [9]:
cleaned_sales = sales.merge(products, on='ProductKey', how='left')
cleaned_sales = cleaned_sales.merge(customers, on='CustomerKey', how='left')
cleaned_sales = cleaned_sales.merge(
    stores,
    on='StoreKey',
    how='left',
    suffixes=('_Customer', '_Store'),
)
cleaned_sales = cleaned_sales.merge(
    exchange_rates,
    how='left',
    left_on=['Order Date', 'Currency Code'],
    right_on=['Date', 'Currency'],
)

print('Rows in cleaned dataset:', len(cleaned_sales))

Rows in cleaned dataset: 62884


## 10. Create sales and profit columns

In [10]:
cleaned_sales['Sales USD'] = cleaned_sales['Quantity'] * cleaned_sales['Unit Price USD']
cleaned_sales['Cost USD'] = cleaned_sales['Quantity'] * cleaned_sales['Unit Cost USD']
cleaned_sales['Profit USD'] = cleaned_sales['Sales USD'] - cleaned_sales['Cost USD']

## 11. Create the required derived columns

In [11]:
cleaned_sales['Year'] = cleaned_sales['Order Date'].dt.year
cleaned_sales['Quarter'] = cleaned_sales['Order Date'].dt.quarter
cleaned_sales['Month'] = cleaned_sales['Order Date'].dt.month
cleaned_sales['Month Name'] = cleaned_sales['Order Date'].dt.month_name()
cleaned_sales['Day of Week'] = cleaned_sales['Order Date'].dt.day_name()

cleaned_sales['Profit Margin %'] = np.where(
    cleaned_sales['Sales USD'] > 0,
    (cleaned_sales['Profit USD'] / cleaned_sales['Sales USD']) * 100,
    0,
)

cleaned_sales['Sales Category'] = pd.cut(
    cleaned_sales['Sales USD'],
    bins=[-np.inf, 500, 1000, np.inf],
    labels=['Low', 'Medium', 'High'],
)

display(cleaned_sales[[
    'Order Date', 'Year', 'Quarter', 'Month', 'Month Name',
    'Day of Week', 'Profit Margin %', 'Sales Category'
]].head())

,Order Date,Year,Quarter,Month,Month Name,Day of Week,Profit Margin %,Sales Category
0,2016-01-01,2016,1,1,January,Friday,54.014706,Low
1,2016-01-01,2016,1,1,January,Friday,66.868852,Medium
2,2016-01-01,2016,1,1,January,Friday,66.867886,Medium
3,2016-01-01,2016,1,1,January,Friday,54.012422,High
4,2016-01-01,2016,1,1,January,Friday,49.018405,Low


## 12. Save the cleaned dataset

Dates are saved in `YYYY-MM-DD` format.

In [12]:
output_file = processed_data_folder / 'cleaned_sales.csv'

cleaned_sales.to_csv(
    output_file,
    index=False,
    date_format='%Y-%m-%d',
)

print('Cleaned dataset saved to:', output_file)

Cleaned dataset saved to: d:\Desktop\Pune Internship\Global-Electronics-Retail-Sales-Analysis-and-Business-Intelligence-Dashboard\data\processed\cleaned_sales.csv
